In [ ]:
display(
    spark.sql(
        """
select * from system.billing.usage limit 2"""
    )
)

account_id,workspace_id,record_id,sku_name,cloud,usage_start_time,usage_end_time,usage_date,custom_tags,usage_unit,usage_quantity,usage_metadata,identity_metadata,record_type,ingestion_date,billing_origin_product,product_features,usage_type
41b9abac-ad59-4b43-b757-405183b9c40c,3025635997916625,JHaFCAitxqZNcL8Q_aPESLyw,ENTERPRISE_ALL_PURPOSE_COMPUTE,AWS,2025-07-22T16:00:00Z,2025-07-22T17:00:00Z,2025-07-22,[object Object],DBU,2.609456586111111111,"0528-203315-fitpbko,,,,m5dn.2xlarge,,,,,,,,,,,,,,,,,,,,,,,,,,",",,",ORIGINAL,2025-07-22,ALL_PURPOSE,",,,false,false,,,,,",COMPUTE_TIME
41b9abac-ad59-4b43-b757-405183b9c40c,5006416539648865,JHaJiSxhH11PIKE9zqcQXCKF,ENTERPRISE_SERVERLESS_SQL_COMPUTE_US_EAST_N_VIRGINIA,AWS,2025-07-22T16:00:00Z,2025-07-22T17:00:00Z,2025-07-22,[object Object],DBU,7.159853333333333333,",,e3febba55ff0f468,,db.medium,,,,,,,,,,,,,,,,,,,,,,,,,,",",,__REDACTED__",ORIGINAL,2025-07-22,SQL,",,,true,true,,,,,",COMPUTE_TIME


In [7]:
display(
    spark.sql(
        """
SELECT
    u.workspace_id,
    u.usage_date AS ds,
    CAST(u.usage_quantity AS DOUBLE) AS dbus,
    CAST(lp.pricing.default * usage_quantity AS DOUBLE) AS cost_at_list_price,
    COALESCE(
      CASE
        WHEN u.custom_tags.Tenant IN ('oh', 'haven') THEN 'oh_haven'
        WHEN u.custom_tags.Tenant = 'oh-epa' THEN 'oh_epa'
        WHEN u.custom_tags.Tenant = 'Texas' THEN 'tx'
        WHEN u.custom_tags.Tenant = 'Insight' THEN 'insight'
        WHEN u.custom_tags.Tenant = 'insight_base_tables' THEN 'insight_base'
        WHEN u.custom_tags.Tenant = 'ca-uw2' THEN 'ca'
        WHEN u.custom_tags.Tenant = 'de_edw' THEN 'de'
        WHEN u.custom_tags.Tenant = 'hedis_ga_dev' THEN 'hedis_ga'
        WHEN u.custom_tags.Tenant = 'hedis_ga_stg' THEN 'hedis_ga'
        WHEN u.custom_tags.Tenant = 'hedis_oz_dev' THEN 'hedis_oz'
        WHEN u.custom_tags.Tenant = 'hedis_nm_dev' THEN 'hedis_nm'
        WHEN u.custom_tags.Tenant = 'MS' THEN 'hedis_ms'
        WHEN u.workspace_id = '67969358316836'
        AND u.custom_tags.Tenant = 'tx' THEN 'insight_tx'
        WHEN u.workspace_id = '5048061916187028'
        AND u.custom_tags.Tenant = 'oz' THEN 'insight_ky'
        WHEN u.workspace_id = '7592211574530171'
        AND u.custom_tags.Tenant = 'tx' THEN 'insight_tx'
        WHEN u.workspace_id = '2762755653818796'
        AND u.custom_tags.Tenant = 'tenant' THEN 'ks'
        WHEN u.workspace_id = '487358706119890'
        AND u.custom_tags.Tenant = 'tenant' THEN 'ks'
        WHEN u.workspace_id = '3114357280267733'
        AND u.custom_tags.Tenant = 'oz' THEN 'tx'
        ELSE LOWER(REPLACE(u.custom_tags.Tenant, '-', '_'))
      END,
      'NULL'
    ) AS tenant,
    CASE
      WHEN CONTAINS(u.sku_name, 'ALL_PURPOSE') THEN 'All Purpose'
      WHEN CONTAINS(u.sku_name, 'JOBS') THEN 'Jobs'
      WHEN CONTAINS(u.sku_name, 'SQL')
      AND NOT CONTAINS(u.sku_name, 'SERVERLESS') THEN 'SQL Compute'
      WHEN CONTAINS(u.sku_name, 'SQL')
      AND CONTAINS(u.sku_name, 'SERVERLESS') THEN 'Serverless SQL Compute'
      WHEN CONTAINS(u.sku_name, 'INFERENCE') THEN 'Model Inference'
      WHEN CONTAINS(u.sku_name, 'DLT') THEN 'Delta Live tables'
      ELSE 'Other'
    END AS sku
  FROM
    system.billing.usage u
    INNER JOIN system.billing.list_prices lp ON u.cloud = lp.cloud
    AND u.sku_name = lp.sku_name
    AND u.usage_start_time >= lp.price_start_time
    AND (
      u.usage_end_time <= lp.price_end_time
      OR lp.price_end_time IS NULL
    )
  WHERE
    u.usage_unit = 'DBU'
    AND NOT (
      u.billing_origin_product == 'LAKEHOUSE_MONITORING'
      OR u.billing_origin_product == 'PREDICTIVE_OPTIMIZATION'
    )
    AND u.usage_date = DATE_SUB(CURRENT_DATE, 1)"""
    )
)

workspace_id,ds,dbus,cost_at_list_price,tenant,sku
7318887597016410,2025-07-21,1.7943403127777777,0.358868,oh_haven,Jobs
7318887597016410,2025-07-21,1.8631601805555555,0.372632,oh_haven,Jobs
3964632188856093,2025-07-21,1.6906772027777779,0.338135,oh_haven,Jobs
3564918100475480,2025-07-21,28.362,5.6724,oh_haven,Jobs
3564918100475480,2025-07-21,40.12896049166667,8.025792,oh_haven,Jobs
7318887597016410,2025-07-21,1.8651376422222221,0.373028,oh_haven,Jobs
3564918100475480,2025-07-21,1.5517034027777779,0.310341,oh_haven,Jobs
7318887597016410,2025-07-21,0.5337775444444445,0.106756,oh_haven,Jobs
7318887597016410,2025-07-21,1.8274083038888889,0.365482,oh_haven,Jobs
7318887597016410,2025-07-21,1.9926353366666667,0.398527,oh_haven,Jobs


In [5]:
display(
    spark.sql(
        """
SELECT
    u.workspace_id,
    u.usage_date AS ds,
    CAST(u.usage_quantity AS DOUBLE) AS dbus,
    CAST(lp.pricing.default * usage_quantity AS DOUBLE) AS cost_at_list_price,
    COALESCE(
      CASE
        WHEN u.custom_tags.Tenant IN ('oh', 'haven') THEN 'oh_haven'
        WHEN u.custom_tags.Tenant = 'oh-epa' THEN 'oh_epa'
        WHEN u.custom_tags.Tenant = 'Texas' THEN 'tx'
        WHEN u.custom_tags.Tenant = 'Insight' THEN 'insight'
        WHEN u.custom_tags.Tenant = 'insight_base_tables' THEN 'insight_base'
        WHEN u.custom_tags.Tenant = 'ca-uw2' THEN 'ca'
        WHEN u.custom_tags.Tenant = 'de_edw' THEN 'de'
        WHEN u.custom_tags.Tenant = 'hedis_ga_dev' THEN 'hedis_ga'
        WHEN u.custom_tags.Tenant = 'hedis_ga_stg' THEN 'hedis_ga'
        WHEN u.custom_tags.Tenant = 'hedis_oz_dev' THEN 'hedis_oz'
        WHEN u.custom_tags.Tenant = 'hedis_nm_dev' THEN 'hedis_nm'
        WHEN u.custom_tags.Tenant = 'MS' THEN 'hedis_ms'
        WHEN u.workspace_id = '67969358316836'
        AND u.custom_tags.Tenant = 'tx' THEN 'insight_tx'
        WHEN u.workspace_id = '5048061916187028'
        AND u.custom_tags.Tenant = 'oz' THEN 'insight_ky'
        WHEN u.workspace_id = '7592211574530171'
        AND u.custom_tags.Tenant = 'tx' THEN 'insight_tx'
        WHEN u.workspace_id = '2762755653818796'
        AND u.custom_tags.Tenant = 'tenant' THEN 'ks'
        WHEN u.workspace_id = '487358706119890'
        AND u.custom_tags.Tenant = 'tenant' THEN 'ks'
        WHEN u.workspace_id = '3114357280267733'
        AND u.custom_tags.Tenant = 'oz' THEN 'tx'
        ELSE LOWER(REPLACE(u.custom_tags.Tenant, '-', '_'))
      END,
      'NULL'
    ) AS tenant,
    CASE
      WHEN CONTAINS(u.sku_name, 'ALL_PURPOSE') THEN 'All Purpose'
      WHEN CONTAINS(u.sku_name, 'JOBS') THEN 'Jobs'
      WHEN CONTAINS(u.sku_name, 'SQL')
      AND NOT CONTAINS(u.sku_name, 'SERVERLESS') THEN 'SQL Compute'
      WHEN CONTAINS(u.sku_name, 'SQL')
      AND CONTAINS(u.sku_name, 'SERVERLESS') THEN 'Serverless SQL Compute'
      WHEN CONTAINS(u.sku_name, 'INFERENCE') THEN 'Model Inference'
      WHEN CONTAINS(u.sku_name, 'DLT') THEN 'Delta Live tables'
      ELSE 'Other'
    END AS sku
  FROM
    system.billing.usage u
    INNER JOIN system.billing.list_prices lp ON u.cloud = lp.cloud
    AND u.sku_name = lp.sku_name
    AND u.usage_start_time >= lp.price_start_time
    AND (
      u.usage_end_time <= lp.price_end_time
      OR lp.price_end_time IS NULL
    )
  WHERE
    u.usage_unit = 'DBU'
    AND NOT (
      u.billing_origin_product == 'LAKEHOUSE_MONITORING'
      OR u.billing_origin_product == 'PREDICTIVE_OPTIMIZATION'
    )
    AND u.usage_date >= LAST_DAY(ADD_MONTHS(CURRENT_DATE, -2)) + INTERVAL 1 DAY
    AND u.usage_date < DATE_TRUNC('month', CURRENT_DATE)"""
    )
)

workspace_id,ds,dbus,cost_at_list_price,tenant,sku
1750434997638272,2025-06-11,2,1.04,NULL,Other
1750434997638272,2025-06-11,2,1.04,NULL,Other
1750434997638272,2025-06-11,2,1.04,NULL,Other
1750434997638272,2025-06-11,2,1.04,NULL,Other
1750434997638272,2025-06-11,2,1.04,NULL,Other
1750434997638272,2025-06-11,2,1.04,NULL,Other
1750434997638272,2025-06-12,2,1.04,NULL,Other
1750434997638272,2025-06-12,2,1.04,NULL,Other
1750434997638272,2025-06-12,2,1.04,NULL,Other
1750434997638272,2025-06-12,2,1.04,NULL,Other


In [ ]:
spark.sql(
    """with price_usage_po as (
  select
    distinct u.account_id,
    u.workspace_id,
    u.sku_name,
    u.cloud
  from
    system.billing.usage u
  where
    u.usage_unit = 'DBU'
    AND u.usage_date >= LAST_DAY(ADD_MONTHS(CURRENT_DATE, -2)) + INTERVAL 1 DAY
    AND u.usage_date < DATE_TRUNC('month', CURRENT_DATE)
    and u.billing_origin_product = 'PREDICTIVE_OPTIMIZATION'
),
price_usage_po_history (
  SELECT
    p.sku_name,
    p.cloud,
    u.*
  FROM
    system.storage.predictive_optimization_operations_history u
    left join price_usage_po p on u.account_id = p.account_id
    and u.workspace_id = p.workspace_id
  WHERE
    to_date(u.start_time) >= LAST_DAY(ADD_MONTHS(CURRENT_DATE, -2)) + INTERVAL 1 DAY
    AND to_date(u.start_time) < DATE_TRUNC('month', CURRENT_DATE)
),
price_usage_po_history_workspace (
  select
    workspace_id,
    catalog_name,
    CAST(u.usage_quantity AS DOUBLE) AS dbus,
    CASE
      WHEN CONTAINS(u.sku_name, 'ALL_PURPOSE') THEN 'All Purpose'
      WHEN CONTAINS(u.sku_name, 'JOBS') THEN 'Jobs'
      WHEN CONTAINS(u.sku_name, 'SQL')
      AND NOT CONTAINS(u.sku_name, 'SERVERLESS') THEN 'SQL Compute'
      WHEN CONTAINS(u.sku_name, 'SQL')
      AND CONTAINS(u.sku_name, 'SERVERLESS') THEN 'Serverless SQL Compute'
      WHEN CONTAINS(u.sku_name, 'INFERENCE') THEN 'Model Inference'
      WHEN CONTAINS(u.sku_name, 'DLT') THEN 'Delta Live tables'
      ELSE 'Other'
    END AS sku,
    to_date(start_time) as ds,
    CAST(lp.pricing.default * usage_quantity AS DOUBLE) AS cost_at_list_price
  from
    price_usage_po_history u
    INNER JOIN system.billing.list_prices lp ON u.cloud = lp.cloud
    AND lp.sku_name = u.sku_name
    AND u.start_time >= lp.price_start_time
    AND (
      u.end_time <= lp.price_end_time
      OR lp.price_end_time IS NULL
    )
),
price_usage_po_history_workspace_catalog (
  select
    c.*,
    b.workspace_name,
    b.env
  from
    price_usage_po_history_workspace c
    left join mgmt_stg.metadata.workspace_id_crosswalk_view b on c.workspace_id = b.workspace_id
),
po_price_usage_catalog_workspace_tenant (
  select
    c.workspace_id,
    c.workspace_name,
    c.ds,
    c.sku,
    c.cost_at_list_price,
    b.tenant,
    c.env,
    "PO" as usage_type
  from
    price_usage_po_history_workspace_catalog c
    left join mgmt_stg.metadata.workspace_id_crosswalk_tenant_env_view b on c.catalog_name = b.catalog_name
    and c.workspace_id = b.workspace_id
),
lm_price_usage AS (
  SELECT
    u.workspace_id,
    u.usage_date AS ds,
    CAST(u.usage_quantity AS DOUBLE) AS dbus,
    CAST(lp.pricing.default * usage_quantity AS DOUBLE) AS cost_at_list_price,
    LOWER(REPLACE(u.custom_tags.Tenant, '-', '_')) AS tenant,
    u.custom_tags.LakehouseMonitoringTableId as table_id,
    CASE
      WHEN CONTAINS(u.sku_name, 'ALL_PURPOSE') THEN 'All Purpose'
      WHEN CONTAINS(u.sku_name, 'JOBS') THEN 'Jobs'
      WHEN CONTAINS(u.sku_name, 'SQL')
      AND NOT CONTAINS(u.sku_name, 'SERVERLESS') THEN 'SQL Compute'
      WHEN CONTAINS(u.sku_name, 'SQL')
      AND CONTAINS(u.sku_name, 'SERVERLESS') THEN 'Serverless SQL Compute'
      WHEN CONTAINS(u.sku_name, 'INFERENCE') THEN 'Model Inference'
      WHEN CONTAINS(u.sku_name, 'DLT') THEN 'Delta Live tables'
      ELSE 'Other'
    END AS sku
  FROM
    system.billing.usage u
    INNER JOIN system.billing.list_prices lp ON u.cloud = lp.cloud
    AND u.sku_name = lp.sku_name
    AND u.usage_start_time >= lp.price_start_time
    AND (
      u.usage_end_time <= lp.price_end_time
      OR lp.price_end_time IS NULL
    )
  WHERE
    u.usage_unit = 'DBU'
    AND u.billing_origin_product = 'LAKEHOUSE_MONITORING'
    AND u.usage_date >= LAST_DAY(ADD_MONTHS(CURRENT_DATE, -2)) + INTERVAL 1 DAY
    AND u.usage_date < DATE_TRUNC('month', CURRENT_DATE)
),
lm_table_metada (
  select
    t.table_catalog,
    t.table_type,
    coalesce(
      split(
        coalesce(t.storage_sub_directory, t.storage_path),
        "tables/"
      ) [1],
      "NA"
    ) as table_id
  from
    system.information_schema.tables t
),
lm_price_usage_catalog (
  select
    u.*,
    m.table_catalog
  from
    lm_price_usage u
    left join lm_table_metada m on m.table_id = u.table_id
),
lm_price_usage_catalog_workspace (
  select
    c.*,
    b.workspace_name,
    b.env
  from
    lm_price_usage_catalog c
    left join mgmt_stg.metadata.workspace_id_crosswalk_view b on c.workspace_id = b.workspace_id
),
lm_price_usage_catalog_workspace_tenant (
  select
    c.workspace_id,
    c.workspace_name,
    c.ds,
    c.sku,
    c.cost_at_list_price,
    b.tenant,
    c.env,
    "LM" as usage_type
  from
    lm_price_usage_catalog_workspace c
    left join mgmt_stg.metadata.workspace_id_crosswalk_tenant_env_view b on c.table_catalog = b.catalog_name
    and c.workspace_id = b.workspace_id
),
rest_price_usage AS (
  SELECT
    u.workspace_id,
    u.usage_date AS ds,
    CAST(u.usage_quantity AS DOUBLE) AS dbus,
    CAST(lp.pricing.default * usage_quantity AS DOUBLE) AS cost_at_list_price,
    COALESCE(
      CASE
        WHEN u.custom_tags.Tenant IN ('oh', 'haven') THEN 'oh_haven'
        WHEN u.custom_tags.Tenant = 'oh-epa' THEN 'oh_epa'
        WHEN u.custom_tags.Tenant = 'Texas' THEN 'tx'
        WHEN u.custom_tags.Tenant = 'Insight' THEN 'insight'
        WHEN u.custom_tags.Tenant = 'insight_base_tables' THEN 'insight_base'
        WHEN u.custom_tags.Tenant = 'ca-uw2' THEN 'ca'
        WHEN u.custom_tags.Tenant = 'de_edw' THEN 'de'
        WHEN u.custom_tags.Tenant = 'hedis_ga_dev' THEN 'hedis_ga'
        WHEN u.custom_tags.Tenant = 'hedis_ga_stg' THEN 'hedis_ga'
        WHEN u.custom_tags.Tenant = 'hedis_oz_dev' THEN 'hedis_oz'
        WHEN u.custom_tags.Tenant = 'hedis_nm_dev' THEN 'hedis_nm'
        WHEN u.custom_tags.Tenant = 'MS' THEN 'hedis_ms'
        WHEN u.workspace_id = '67969358316836'
        AND u.custom_tags.Tenant = 'tx' THEN 'insight_tx'
        WHEN u.workspace_id = '5048061916187028'
        AND u.custom_tags.Tenant = 'oz' THEN 'insight_ky'
        WHEN u.workspace_id = '7592211574530171'
        AND u.custom_tags.Tenant = 'tx' THEN 'insight_tx'
        WHEN u.workspace_id = '2762755653818796'
        AND u.custom_tags.Tenant = 'tenant' THEN 'ks'
        WHEN u.workspace_id = '487358706119890'
        AND u.custom_tags.Tenant = 'tenant' THEN 'ks'
        WHEN u.workspace_id = '3114357280267733'
        AND u.custom_tags.Tenant = 'oz' THEN 'tx'
        ELSE LOWER(REPLACE(u.custom_tags.Tenant, '-', '_'))
      END,
      'NULL'
    ) AS tenant,
    CASE
      WHEN CONTAINS(u.sku_name, 'ALL_PURPOSE') THEN 'All Purpose'
      WHEN CONTAINS(u.sku_name, 'JOBS') THEN 'Jobs'
      WHEN CONTAINS(u.sku_name, 'SQL')
      AND NOT CONTAINS(u.sku_name, 'SERVERLESS') THEN 'SQL Compute'
      WHEN CONTAINS(u.sku_name, 'SQL')
      AND CONTAINS(u.sku_name, 'SERVERLESS') THEN 'Serverless SQL Compute'
      WHEN CONTAINS(u.sku_name, 'INFERENCE') THEN 'Model Inference'
      WHEN CONTAINS(u.sku_name, 'DLT') THEN 'Delta Live tables'
      ELSE 'Other'
    END AS sku
  FROM
    system.billing.usage u
    INNER JOIN system.billing.list_prices lp ON u.cloud = lp.cloud
    AND u.sku_name = lp.sku_name
    AND u.usage_start_time >= lp.price_start_time
    AND (
      u.usage_end_time <= lp.price_end_time
      OR lp.price_end_time IS NULL
    )
  WHERE
    u.usage_unit = 'DBU'
    AND NOT (
      u.billing_origin_product == 'LAKEHOUSE_MONITORING'
      OR u.billing_origin_product == 'PREDICTIVE_OPTIMIZATION'
    )
    AND u.usage_date >= LAST_DAY(ADD_MONTHS(CURRENT_DATE, -2)) + INTERVAL 1 DAY
    AND u.usage_date < DATE_TRUNC('month', CURRENT_DATE)
),
rest_price_usage_catalog_workspace (
  select
    c.*,
    b.workspace_name,
    b.env
  from
    rest_price_usage c
    left join mgmt_stg.metadata.workspace_id_crosswalk_view b on c.workspace_id = b.workspace_id
),
rest_price_usage_catalog_workspace_tenant (
  select
    c.workspace_id,
    c.workspace_name,
    c.ds,
    c.sku,
    c.cost_at_list_price,
    b.tenant,
    c.env,
    "REST" as usage_type
  from
    rest_price_usage_catalog_workspace c
    left join mgmt_stg.metadata.workspace_id_crosswalk_tenant_env_view b on c.tenant = b.tenant
    and c.workspace_id = b.workspace_id
),
price_usage_tenant (
  select
    *
  from
    rest_price_usage_catalog_workspace_tenant
  union all
  select
    *
  from
    lm_price_usage_catalog_workspace_tenant
  union all
  select
    *
  from
    po_price_usage_catalog_workspace_tenant
),
usage_pivot AS (
  SELECT
    tenant,
    ds as usage_date,
    DATE_FORMAT(ds, 'MMM') AS usage_month,
    workspace_id,
    workspace_name,
    env,
    sku,
    usage_type,
    SUM(cost_at_list_price) AS cost
  FROM
    price_usage_tenant
  GROUP BY
    tenant,
    ds,
    DATE_FORMAT(ds, 'MMM'),
    workspace_id,
    workspace_name,
    env,
    sku,
    usage_type
  ORDER BY
    workspace_id,
    ds
),
usage_per_workspace_per_sku AS (
      SELECT
        *
      FROM
        usage_pivot PIVOT (
          SUM(cost) FOR (sku) IN (
            'All Purpose',
            'Jobs',
            'SQL Compute',
            'Serverless SQL Compute',
            'Model Inference',
            'Delta Live tables',
            'Other'
          )
        )
    )
SELECT
      u.tenant,
      u.env,
      u.workspace_name,
      u.usage_type,
      u.usage_month,
      u.workspace_id,
      CAST(SUM(COALESCE(u.`All Purpose`, 0) + COALESCE(u.`Jobs`, 0) + COALESCE(u.`SQL Compute`, 0)
      + COALESCE(u.`Serverless SQL Compute`, 0) + COALESCE(u.`Model Inference`, 0)
      + COALESCE(u.`Delta Live tables`, 0) + COALESCE(u.`Other`, 0)) AS DECIMAL(10,4)) AS `Total Dollar Spent`,
      CAST(COALESCE(SUM(u.`All Purpose`), 0) AS DECIMAL(10,4)) AS `All Purpose`,
      CAST(COALESCE(SUM(u.`Jobs`), 0) AS DECIMAL(10,4)) AS `Jobs`,
      CAST(COALESCE(SUM(u.`SQL Compute`), 0) AS DECIMAL(10,4)) AS `SQL Compute`,
      CAST(COALESCE(SUM(u.`Serverless SQL Compute`), 0) AS DECIMAL(10,4)) AS `Serverless SQL Compute`,
      CAST(COALESCE(SUM(u.`Model Inference`), 0) AS DECIMAL(10,4)) AS `Model Inference`,
      CAST(COALESCE(SUM(u.`Delta Live tables`), 0) AS DECIMAL(10,4)) AS `Delta Live tables`,
      CAST(COALESCE(SUM(u.`Other`), 0) AS DECIMAL(10,4)) AS `Other`
    FROM
      usage_per_workspace_per_sku u
    GROUP BY
      u.tenant,
      u.env,
      u.workspace_name,
      u.usage_type,
      u.usage_month,
      u.workspace_id
    ORDER BY
      tenant, workspace_name"""
)